In [ ]:
from langgraph.graph import StateGraph
from langchain_core.messages import AnyMessage,HumanMessage,AIMessage
from typing_extensions import TypedDict,List,Union
from dotenv import find_dotenv, load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langchain_google_vertexai import ChatVertexAI
import vertexai

In [ ]:
import os
load_dotenv(find_dotenv())

In [ ]:
#openai_api_key = os.environ["OPENAI_API_KEY"]


#print("API Key Loaded:", os.environ.get("OPENAI_API_KEY"),"Got it")

In [ ]:
gemini_api_key = os.environ["GEMINI_API_KEY"]
google_application_credentials = os.environ["GOOGLE_APPLICATION_CREDENTIALS"]

print("API Key Loaded:", os.environ.get("GEMINI_API_KEY"))
print("Google Application Credentials Loaded:", os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"))

GOOGLE_CLOUD_PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
print("Google Cloud Project ID:", os.environ.get("GOOGLE_CLOUD_PROJECT"))



In [ ]:
class AgentState(TypedDict):
    messages: List[Union[HumanMessage,AIMessage]]
    
graph_builder = StateGraph(AgentState)

In [ ]:
#llm = init_chat_model("openai:gpt-4o-mini")
#llm = init_chat_model("gemini-1.5-pro", model_provider="google_vertexai")
vertexai.init(project=GOOGLE_CLOUD_PROJECT, location="us-central1")
llm = ChatVertexAI(model_name="gemini-1.5-flash-001")

In [ ]:
def chatbot(state: AgentState) -> AgentState:
    """A simple chatbot that echoes user input."""
    # Get the last message from the state
    human_message = state["messages"]
    
    # Generate a response using the LLM
    response = llm.invoke(human_message)
    
    # Add the response to the messages
    state["messages"].append(AIMessage(content=response.content))
    
    print("LLM Response:", response.content)
    
    return state



In [ ]:
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
conversation_history = []

#user_input = input("User: ")
#while user_input.lower() != "exit":

user_input = "Hey, What is the capital of India?"
conversation_history.append(HumanMessage(content=user_input))
response = graph.invoke({"messages": conversation_history})
conversation_history = response["messages"]
print (conversation_history[-1].content)
#   user_input = input("User: ")
